This nb digs deeper into the mechanics of logging with `mlflow`.

## Set Up `mlflow`

In [1]:
import mlflow
from pathlib import Path

# point the tracking server to the local mlflow server
# if you don't do this, mlflow will log to a local directory called mlruns
mlflow.set_tracking_uri("http://127.0.0.1:5000")

# set a new experiment
mlflow.set_experiment("mlflow-logging-demo")

# inspect the experiment
experiment = mlflow.get_experiment_by_name("mlflow-logging-demo")
print(f"Experiment_id: {experiment.experiment_id}")
print(f"Artifact Location: {experiment.artifact_location}")
print(f"Tags: {experiment.tags}")
print(f"Lifecycle_stage: {experiment.lifecycle_stage}")

2026/11/04 10:12:34 INFO mlflow.tracking.fluent: Experiment with name 'mlflow-logging-demo' does not exist. Creating a new experiment.


Experiment_id: 2
Artifact Location: mlflow-artifacts:/2
Tags: {}
Lifecycle_stage: active


## Prep Data

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml

# load the adult dataset from openml
# mixed data types, some missing values, and a binary target
df = fetch_openml("adult", version=2, as_frame=True, parser="pandas").frame

print(f"Shape:  {df.shape}")
print(f"Target: 'class':{df['class'].unique()}")
df.head()

Shape:  (48842, 15)
Target: 'class':['<=50K', '>50K']
Categories (2, str): ['<=50K', '>50K']


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,<=50K


In [ ]:
# clean missing values -- lazily since we're here for mlflow
df1 = df.dropna()
print(f"Shape after cleaning: {df1.shape}")

Shape after cleaning: (45222, 15)


In [18]:
# binarize the target variable
df1["class"] = df1["class"].apply(lambda x: 1 if x == ">50K" else 0)

In [19]:
features = df1.columns[:-1]  # all feature cols
target = 'class'

def train_test_split(df, test_size=0.2, random_state=42):
    """Split the dataframe into train and test sets."""
    df = df.sample(frac=1, random_state=random_state).reset_index(drop=True)
    test_size = int(len(df) * test_size)
    train_df = df[:-test_size]
    test_df = df[-test_size:]
    return train_df, test_df

train_df, test_df = train_test_split(df1, test_size=0.2, random_state=42)

## Fit Model

In [23]:
from xgboost import XGBClassifier

### Two feature sets

We'll fit the same model twice and change only the inputs:
- **`all_features`**: every column (numeric and categorical)
- **`numeric_subset`**: five numeric columns only

In [20]:
ALL_FEATURES = list(features)
NUMERIC_SUBSET = ["age", "education-num", "hours-per-week", "capital-gain", "capital-loss"]

print(f"all_features:   {len(ALL_FEATURES)} columns")
print(f"numeric_subset: {len(NUMERIC_SUBSET)} columns")

all_features:   14 columns
numeric_subset: 5 columns


## Log a run

Each call to `mlflow.start_run()` creates one **run**, and everything logged inside the `with` block is attached to it. MLflow stores different kinds of information in different ways:

| What | API | Stored as |
|---|---|---|
| Hyperparameters | `log_params` | key/value rows (strings) in the backend store |
| Metrics | `log_metrics` | numeric rows in the backend store, with step and timestamp |
| Tags | `set_tags` | key/value rows, for filtering and annotation |
| Dataset references | `log_input` | name, digest, schema, and profile in the backend store. **The data itself is not copied.** |
| Files (JSON, tables, plots) | `log_dict`, `log_table`, `log_artifact` | files in the artifact store |
| Model | `mlflow.pyfunc.log_model` | a model directory in the artifact store, plus a logged-model record in the backend store |

The first four are small and queryable, so they live in the database. Anything file-shaped goes to the artifact store.

In [ ]:
from __future__ import annotations

import tempfile

import numpy as np
from mlflow.models import infer_signature
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score


# thin wrapper around the XGBClassifier to restore categorical dtypes from strings
class CategoricalXGBClassifier(mlflow.pyfunc.PythonModel):
    """Serve an XGBoost classifier from raw string inputs by restoring categorical dtypes."""

    def __init__(self, categories: dict[str, list[str]]) -> None:
        self.categories = categories

    def load_context(self, context) -> None:
        """Load the saved XGBoost model from the logged artifacts."""
        self.model = XGBClassifier(enable_categorical=True)
        self.model.load_model(context.artifacts["xgb_model"])

    def predict(self, context, model_input: pd.DataFrame, params: dict | None = None) -> np.ndarray:
        """Cast string columns to their training categories, then predict class labels."""
        X = model_input.copy()
        for col, cats in self.categories.items():
            X[col] = pd.Categorical(X[col], categories=cats)
        return self.model.predict(X)


def fit_and_log(run_name: str, cols: list[str], **xgb_overrides) -> str:
    """Fit an XGBoost classifier on `cols` and log it to a new MLflow run."""
    # constant params
    params = {
        "n_estimators": 200,
        "max_depth": 4,
        "learning_rate": 0.1,
        "enable_categorical": True,
        "tree_method": "hist",
        "random_state": 42,
        **xgb_overrides,
    }
    X_train, y_train = train_df[cols], train_df[target]
    X_test, y_test = test_df[cols], test_df[target]

    # with ... context manager logs to specified run and closes it when done
    with mlflow.start_run(run_name=run_name) as run:
        # can define tags for runs, which facilitates searching and filtering later
        mlflow.set_tag("feature_set", run_name)
        mlflow.log_params(params)
        mlflow.log_param("features", ",".join(cols))

        # record which data was used (schema + digest, not the rows themselves)
        train_ds = mlflow.data.from_pandas(train_df[[*cols, target]], name="adult-train", targets=target)
        test_ds = mlflow.data.from_pandas(test_df[[*cols, target]], name="adult-test", targets=target)
        mlflow.log_input(train_ds, context="training")
        mlflow.log_input(test_ds, context="evaluation")

        model = XGBClassifier(**params).fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]
        pred = (proba > 0.5).astype(int)

        # define metrics to log -- passable as a dict or as keyword args
        mlflow.log_metrics(
            {
                "accuracy": accuracy_score(y_test, pred),
                "f1": f1_score(y_test, pred),
                "roc_auc": roc_auc_score(y_test, proba),
            }
        )

        # files go to the artifact store
        importances = dict(zip(cols, map(float, model.feature_importances_)))
        mlflow.log_dict(importances, "feature_importance.json")
        preview = X_test.head(200).assign(actual=y_test.head(200), predicted=pred[:200])
        mlflow.log_table(preview, "test_predictions.json")

        # the model accepts raw strings; the wrapper restores the training categories
        cat_cols = [c for c in cols if isinstance(X_train[c].dtype, pd.CategoricalDtype)]
        categories = {c: list(X_train[c].cat.categories) for c in cat_cols}
        raw_test = X_test.astype({c: "str" for c in cat_cols})  # cast to str for serving

        with tempfile.TemporaryDirectory() as tmp:
            model_path = str(Path(tmp) / "xgb_model.ubj")
            model.save_model(model_path)
            mlflow.pyfunc.log_model(
                name="model",
                python_model=CategoricalXGBClassifier(categories),
                artifacts={"xgb_model": model_path},
                signature=infer_signature(raw_test, pred),
                input_example=raw_test.head(5),
            )
    return run.info.run_id

**Notes:**\
What cell 14 logs, in the order it happens:
- `start_run` opens a run, and everything logged inside the `with` block is attached to it. The run closes (and is marked `FINISHED`) when the block exits.
- `set_tag` and `log_params` record searchable labels and the hyperparameters. Both are stored as strings.
- `from_pandas` + `log_input` link the train and test data to the run. Only the name, digest, schema, and profile are stored, not the rows. Digests let you tell whether two runs saw the same data.
- `log_metrics` stores numeric results, which the UI can chart and compare across runs.
- `log_dict` and `log_table` write small files (JSON) to the artifact store.

`log_model`:
- `CategoricalXGBClassifier` is a `pyfunc` wrapper. JSON can't represent pandas categoricals, so the model's input example (and any serving request) arrives as plain strings. The wrapper casts those columns back to the training categories before predicting. That lets the saved model score raw string input, which a bare XGBoost model can't do here.
- `infer_signature` infers the model signature from sample inputs and outputs: the column names and types the model expects, and what it returns. MLflow enforces the signature at predict time and shows it in the UI. We infer it from the raw string version of the data, since that is what callers will send.
- `input_example` saves the passed dataset as JSON, then loads and scores it with the model to capture dependencies and check whether the example can be served.
- `artifacts={"xgb_model": ...}` copies the saved XGBoost file into the model directory, so the logged model is self-contained.

In [26]:
run_all = fit_and_log("all_features", ALL_FEATURES)
run_subset = fit_and_log("numeric_subset", NUMERIC_SUBSET)
print(f"all_features run:   {run_all}")
print(f"numeric_subset run: {run_subset}")

c:\Users\bakrefens\repos\mlflow-demos\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
c:\Users\bakrefens\repos\mlflow-demos\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input d

2026/11/04 11:31:39 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run all_features at: http://127.0.0.1:5000/#/experiments/2/runs/a16e4e483b054dae9216057d034718fb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


c:\Users\bakrefens\repos\mlflow-demos\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
c:\Users\bakrefens\repos\mlflow-demos\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input d

2026/11/04 11:31:44 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run numeric_subset at: http://127.0.0.1:5000/#/experiments/2/runs/4b2e3ff62cfd44abbb31298232132497
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
all_features run:   a16e4e483b054dae9216057d034718fb
numeric_subset run: 4b2e3ff62cfd44abbb31298232132497


## Compare the runs

**In the UI** (`http://127.0.0.1:5000`): open the `mlflow-logging-demo` experiment, tick both runs, and click **Compare**. You get side-by-side params, metrics, and charts. Each run's **Artifacts** tab holds the model, `feature_importance.json`, and the predictions table.

**In code**, `search_runs` returns runs as a DataFrame, so comparisons are ordinary pandas.

In [ ]:
# runs is a pandas dataframs
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"attributes.run_id IN ('{run_all}', '{run_subset}')",  # SQL-like, not SQL -- can be tricky
)

metric_cols = ["metrics.accuracy", "metrics.f1", "metrics.roc_auc"]
comparison = runs.set_index("tags.feature_set")[metric_cols].T
comparison["delta (all - subset)"] = comparison["all_features"] - comparison["numeric_subset"]
comparison

tags.feature_set,numeric_subset,all_features,delta (all - subset)
metrics.accuracy,0.835803,0.868753,0.032950
metrics.f1,0.582513,0.710699,0.128186
metrics.roc_auc,0.873758,0.926507,0.052749


In [ ]:
# artifacts are retrieved by run id, so we can compare files logged by each run
# comparing feature importances:
importance = pd.DataFrame(
    {
        name: mlflow.artifacts.load_dict(f"runs:/{run_id}/feature_importance.json")
        for name, run_id in [("all_features", run_all), ("numeric_subset", run_subset)]
    }
)
importance.sort_values("all_features", ascending=False)

,all_features,numeric_subset
relationship,0.335936,NaN
marital-status,0.214685,NaN
capital-gain,0.089791,0.241948
education,0.088828,NaN
occupation,0.069169,NaN
capital-loss,0.047902,0.104169
age,0.041503,0.194212
hours-per-week,0.037864,0.116940
sex,0.021918,NaN
native-country,0.016289,NaN


## How MLflow stores what we logged

There are two stores, and the server brokers access to both:

- **Backend store** (`mlflow.db`, SQLite): runs, params, metrics, tags, dataset references, and logged-model records, all as rows.
- **Artifact store** (`mlartifacts/<experiment_id>/...`): files such as the model, JSON, and tables.

The cell below inspects the run, its artifacts, and the record of its data using the client API.

In [30]:
from mlflow import MlflowClient

client = MlflowClient()
run = client.get_run(run_all)

# 1. client API: the run record
print("status:       ", run.info.status)
print("artifact_uri: ", run.info.artifact_uri)
print("params:       ", dict(list(run.data.params.items())[:4]), "...")
print("metrics:      ", run.data.metrics)
print("datasets:     ", [(d.dataset.name, d.dataset.digest, dict(d.tags)) for d in run.inputs.dataset_inputs])

# 2. client API: the artifact listing (files live in the artifact store)
print("\nartifacts:")
for a in client.list_artifacts(run_all):
    print(f"  {a.path}{'/' if a.is_dir else ''}")

# the dataset reference holds a schema and a profile, not the rows
ds = run.inputs.dataset_inputs[0].dataset
print("\nschema (truncated): ", ds.schema[:200], "...")
print("profile:            ", ds.profile)

status:        FINISHED
artifact_uri:  mlflow-artifacts:/2/a16e4e483b054dae9216057d034718fb/artifacts
params:        {'n_estimators': '200', 'max_depth': '4', 'learning_rate': '0.1', 'enable_categorical': 'True'} ...
metrics:       {'accuracy': 0.8687527642636002, 'f1': 0.7106994881793809, 'roc_auc': 0.9265072873209181}
datasets:      [('adult-train', '0cab7d65', {('key', 'mlflow.data.context'): ('value', 'training')}), ('adult-test', '27cdf7d4', {('key', 'mlflow.data.context'): ('value', 'evaluation')})]

artifacts:
  feature_importance.json
  test_predictions.json

schema (truncated):  {"mlflow_colspec": [{"type": "long", "name": "age", "required": true}, {"type": "string", "name": "workclass", "required": true}, {"type": "long", "name": "fnlwgt", "required": true}, {"type": "string ...
profile:             {"num_rows": 36178, "num_elements": 542670}


## Autolog: what is tracked for free

`mlflow.xgboost.autolog()` patches XGBoost so that training logs to MLflow without explicit calls. We fit the same model on all features with autolog on, then diff what it recorded against our manual `all_features` run.

We pass `eval_set` because autolog records per-iteration metrics only for datasets it can evaluate during training.

In [ ]:
mlflow.xgboost.autolog()

autolog_params = {
    "n_estimators": 200,
    "max_depth": 4,
    "learning_rate": 0.1,
    "enable_categorical": True,
    "tree_method": "hist",
    "random_state": 42,
}

# no log_* calls here: autolog records everything from the fit itself
with mlflow.start_run(run_name="autolog_all_features") as autolog_run:
    XGBClassifier(**autolog_params).fit(
        train_df[ALL_FEATURES],
        train_df[target],
        eval_set=[(test_df[ALL_FEATURES], test_df[target])],  # expected by autolog
        verbose=False,
    )

mlflow.xgboost.autolog(disable=True)  # don't leave autolog on for later cells
run_autolog = autolog_run.info.run_id
print(f"autolog run: {run_autolog}")

2026/11/04 12:52:55 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\bakrefens\repos\mlflow-demos\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/11/04 12:52:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/04 12:52:58 WARNING mlfl

🏃 View run autolog_all_features at: http://127.0.0.1:5000/#/experiments/2/runs/9f41c04b87d040a68169f996a9441396
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
autolog run: 9f41c04b87d040a68169f996a9441396


Looking at the metrics of this run in the `mlflow` server UI, you will see a loss-curve that plots the `eval_set` log-likelihood loss at each step of model training (i.e. after each subtree is fit). Up to 200, since that is the max we gave for `n_estimators`. Notably, it doesn't include final accuracy, f1, ROC-AUC metrics by default. So, to track these we would have to declare these like before.

### Diff the two runs

For each category of logged information, the cell below shows what both runs recorded, what only our manual run recorded, and what only autolog recorded. You can also tick both runs in the UI and click **Compare** to see the same differences.

In [35]:
import textwrap


def logged_inventory(run_id: str) -> dict[str, set[str]]:
    """List the names of everything logged to a run, grouped by category."""
    r = client.get_run(run_id)
    models = mlflow.search_logged_models([experiment.experiment_id], output_format="list")
    return {
        "params": set(r.data.params),
        "metrics": set(r.data.metrics),
        "custom tags": {k for k in r.data.tags if not k.startswith("mlflow.")},
        "datasets": {
            f"{d.dataset.name} ({', '.join(t.value for t in d.tags)})" for d in r.inputs.dataset_inputs
        },
        "artifacts": {a.path for a in client.list_artifacts(run_id)},
        "models": {m.name for m in models if m.source_run_id == run_id},
    }


manual, auto = logged_inventory(run_all), logged_inventory(run_autolog)


def show(label: str, names: set[str]) -> None:
    """Print a labelled, wrapped list of names."""
    body = ", ".join(sorted(names)) or "-"
    print(textwrap.fill(f"  {label:<13}({len(names)}) {body}", width=110, subsequent_indent=" " * 18))


for category in manual:
    print(f"\n{category.upper()}")
    show("both:", manual[category] & auto[category])
    show("manual only:", manual[category] - auto[category])
    show("autolog only:", auto[category] - manual[category])


PARAMS
  both:        (4) learning_rate, max_depth, random_state, tree_method
  manual only: (3) enable_categorical, features, n_estimators
  autolog only:(33) base_score, booster, colsample_bylevel, colsample_bynode, colsample_bytree, custom_metric,
                  device, early_stopping_rounds, eval_metric, gamma, grow_policy, interaction_constraints,
                  max_bin, max_cat_threshold, max_cat_to_onehot, max_delta_step, max_leaves, maximize,
                  min_child_weight, monotone_constraints, multi_strategy, n_jobs, num_boost_round,
                  num_parallel_tree, objective, reg_alpha, reg_lambda, sampling_method, scale_pos_weight,
                  subsample, validate_parameters, verbose_eval, verbosity

METRICS
  both:        (0) -
  manual only: (3) accuracy, f1, roc_auc
  autolog only:(1) validation_0-logloss

CUSTOM TAGS
  both:        (0) -
  manual only: (1) feature_set
  autolog only:(0) -

DATASETS
  both:        (0) -
  manual only: (2) adult-test (